# Jak właściwie muzykę charakteryzować?
Potrzebujemy zestawu cech które potem będziemy konkatenować i zamieniać na wektor danych
## Cechy barwowe - brzmienie
### MFCC (Mel-Frequency Cepstral Coefficients)

### Spectral Contrast

### Spectral Centroid

### Spectral Rolloff

### Zero Crossing Rate

## Cechy harmoniczne - tony, emocje
### Chroma Features

## Cechy rytmiczne 
### Tempo - bpm 

### Tempogram



# Ładowanie plików .mp3 
Funkcja ```librosa.load()``` Bierze ścieżke do pliku .mp3, podajemy mu sampling rate (22050 - to wartość w Hz, czyli na sekundę bierzemy 22050 punktów). Dla uproszczenia konwertujemy też sygnał na mono. Niestety, ale dla typowej piosenki wczytywanie trwa prawie minutę, więc spróbujemy innego rozwiązania

In [ ]:
import librosa
import numpy as np
import os
import glob
import pickle
import subprocess
import time
import scipy.fftpack

In [20]:
# y, sr = librosa.load("data/Clipse-The_Birds_Don't_Sing.mp3", sr=22050, mono=True)

Robimy to samo - 22050Hz oraz dźwięk mono, ale używając ffmpeg, który jest napisany i zoptymalizowany w czystym C - to bardzo przyspiesza proces wczytywania. Przy okazji opakuje to w funkcję, która wszystkie pliki .mp3 z folderu data zapisze i wyeksportuje do pliku na którym potem będziemy pracować

In [21]:
def load_audio(file_path, sr=22050):
    command = [
        'ffmpeg',
        '-i', file_path,        # Plik wejściowy
        '-f', 'f32le',          # Format wyjściowy: float 32-bit 
        '-ac', '1',             # Audio Channels: 1 (Mono) 
        '-ar', str(sr),         # Audio Rate: docelowe próbkowanie
        '-acodec', 'pcm_f32le', # Kodek PCM
        '-'                     # Wyjście na standardowe wyjście (pipe) zamiast do pliku
    ]

    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, bufsize=10**8)
    stdout_data, _ = process.communicate()

    # Zamieniamy surowe bajty na tablicę numpy
    audio_array = np.frombuffer(stdout_data, dtype=np.float32)
    
    return audio_array

#Test działania funkcji dla pojedynczego pliku 

y = load_audio("data/Clipse-The_Birds_Don't_Sing.mp3")
print(f"Wczytano {len(y)} próbek.")


def preprocess_dataset(input_folder, output_file, target_sr=22050, album_name=None):
    files = glob.glob(os.path.join(input_folder, "*.mp3"))
    dataset = {}

    for path in files:
        filename = os.path.basename(path)
        try:
            audio_data = load_audio(path, sr=target_sr)
            dataset[filename] = audio_data            
        except Exception as e:
            print(f"Błąd przy pliku {filename}: {e}")

    with open(output_file, 'wb') as f:
        pickle.dump(dataset, f)

preprocess_dataset('data', 'dataset.pkl')

Wczytano 5304832 próbek.


# MFCC (Mel Frequency Cepstral Coefficients)
## Wstęp
Komputery „słyszą” częstotliwości liniowo (różnica między 100 Hz a 200 Hz jest dla nich taka sama jak między 10000 Hz a 10100 Hz). Ludzkie ucho działa inaczej – logarytmicznie. Jesteśmy bardzo czuli na zmiany w niskich tonach, ale w wysokich rozróżniamy je znacznie gorzej. MFCC powstało, aby przekształcić surowy sygnał w coś, co przypomina percepcję człowieka.

## Pre-emphasis
Sygnały audio mają tendencję do posiadania mniejszej energii w wysokich częstotliwościach. Preemfaza to filtr, który podbija wysokie tony, aby zrównoważyć widmo i poprawić stosunek sygnału do szumu. Liczymy to ze wzoru:

$$ y(t) = x(t) - \alpha x(t-1) $$

, gdzie $x(t)$ to wave dźwięku, a parametr $\alpha$ jest bliska 1 (np. 0.97)

## Windowing
Zaczynamy od pojęcia **Windowingu** - dzielimy sobie utwór na krótkie (20-40ms z krokiem 10ms, czyli przedziały się nakrywają) przedziały i stosujemy na nich FFT. Nie ma sensu na dużych przedziałach bo utwory są zbyt zmienne w czasie, a FFT zakłada że sygnał jest stały.
Te okienka będą ucięte na krawędziach, co jest bardzo niekorzystne dla FFT więc chcemy te brzegi wygładzić. Nałożymy na te ramki funkcję Hamminga:

![hamming](images/Hanning.png)

## DFT 
Teraz musimy przejść z dziedziny czasu (amplituda w czasie) do dziedziny częstotliwości (jakie częstotliwości budują ten dźwięk). Obliczamy tzw. periodogram (widmo mocy).

Dla każdej ramki o długości N wykonujemy DFT (Dyskretną Transformacje Fouriera):

$$X[k]=\sum_{n=0}^{N-1} x[n]e^{-j2\pi kn/N}$$

i liczymy spektrum mocy (Power Spectrum):
$$P[k] = \frac{1}{N}|X[k]|^2$$

## Skala Mel
Musimy przeliczyć widmo ze skali Hz na skalę Mel. Skala Mel odzwierciedla to, jak człowiek odbiera wysokość dźwięku.Wzór na konwersję częstotliwości $f$ (w Hz) na Mel ($m$):
$$m = 2595 \log_{10}\left(1 + \frac{f}{700}\right)$$

W praktyce nakładamy na widmo mocy zestaw (bank) trójkątnych filtrów:
- W niskich częstotliwościach filtry są wąskie i gęsto upakowane (wysoka rozdzielczość).
- W wysokich częstotliwościach filtry są szerokie i rzadkie (niska rozdzielczość).

![mel](images/mel_scale.png)

Sumujemy energię pod każdym trójkątem. Jeśli użyjemy 40 filtrów, z tysięcy prążków widma otrzymamy tylko 40 liczb reprezentujących energię w poszczególnych pasmach Mel.

## Logarytmowanie
Bierzemy logarytm z energii obliczonych w poprzednim kroku. Dlaczego? Ponieważ ludzka percepcja głośności również jest logarytmiczna (dlatego używamy decybeli). To sprawia, że cechy są bardziej zbliżone do tego, co "słyszy" mózg, a nie mikrofon.

## Dyskretna Transformacja Kosinusowa (DCT)
Ponieważ filtry w banku Mel (krok 4) zachodzą na siebie, energie w sąsiednich pasmach są ze sobą mocno skorelowane (zawierają podobne informacje). W uczeniu maszynowym wolimy cechy nieskorelowane.

DCT działa podobnie do FFT, ale operuje na liczbach rzeczywistych. Przekształca ono logarytmiczne energie z banku filtrów w cepstrum.

### Szczegóły
Przekształcamy sygnał ($x[n]$) na sumę podstawowych sygnałów ($c[n]$)
$$ x[n] = \sum_{k=0}^{N-1} b_k c_k[n] $$
dla n = 0, 1, ..., N. DCT wykorzystuje się do kompresji, gdzie np. pozbywamy się wszystkich podsygnałów, których wagi $b_k \approx 0$ i transmitujemy tylko resztę

W DCT $b_0 = \frac{1}{N}X^c[0]$, $c_0[n]=1$, $b_k=\frac{1}{N}2X^c[k]$, $c_k[n]=cos(2 \pi (\frac{k+1/2}{2N})n)$ 

$$X^c[k] = \sum _{n=0}^{N-1}x[n]cos(2\pi(\frac{k}{2N})(n+1/2))$$

, w przeciwieństwie do DFT te współczynniki $X^c[k]$ to są liczby rzeczywiste

### MP3
W pliku MP3 kompresja działa następująco:
1. Dzielimy audio na ramki o szerokości 26ms 
2. Dostajemy jakiś długi, podzielony sygnał x[n]
3. Aplikujemy na niego DCT i otrzymujemy współczynniki $X^c[k]$ - tylko one są potrzebne
4. Zachowujemy tylko te współczynniki, które nie są bardzo małe 
5. Zamieniamy je na bity 

## Podsumowanie - Cepstrum
Matematycznie MFCC to:
Widmo (Spectrum) $\to$ Skala Mel $\to$ Logarytm $\to$ Widmo odwrotne (DCT)
W rezultacie otrzymujemy wektor (zazwyczaj 13 liczb na każdą ramkę czasową), który jest niezwykle skompresowanym, ale treściwym opisem dźwięku, idealnym dla sieci neuronowych. Cały ten proces (liczenie widma z widma) - to Cepstrum

![cepstrum](images/cepstrum.png)

In [22]:
def pre_emphasis(signal, alpha=0.97):
    return np.append(signal[0], signal[1:] - alpha * signal[:-1])

def frame_and_window_signal(signal, sr, size=0.025, stride=0.01):
    signal_length = len(signal)
    frame_length = int(round(size * sr))
    frame_step = int(round(stride * sr))
    
    num_frames = int(np.ceil(float(np.abs(signal_length - frame_length)) / frame_step))
    '''
        Padding - chcemy aby ostatnia ramka nie była ucięta oraz nie była krótsza niż poprzednie,
        więc uzupełniamy ją zerami
    '''
    pad_signal_length = num_frames * frame_step + frame_length
    z = np.zeros((pad_signal_length - signal_length))
    pad_signal = np.append(signal, z)
    '''
        Teraz chcielibyśmy wziąć sygnał 1D i zrobić z niego macierz 2D, która zawiera wektory
        odpowiadające każdej ramce
        Zaczynamy od pierwszego kroku - dostajemy macierz z indeksami względnymi (wewnętrznymi):
        [[0, 1, 2],
         [0, 1, 2],
         [0, 1, 2]]
        Takie coś mówi nam, że z każdej ramki bierzemy kolejno jej 0, 1 i 2 element
        Do tego dodajemy indeksy przesunięcia, czyli np. 
        [[0, 0, 0],  
         [2, 2, 2],  
         [4, 4, 4]]
            Tutaj potrzeba transpozycji 
        Po zsumowaniu dostajemy 
        [[0, 1, 2],
         [2, 3, 4],
         [4, 5, 6]]
        Czyli te wskaźniki, których potrzebujemy do stworzenia tej macierzy ramek 
    '''
    indices = np.tile(np.arange(0, frame_length), (num_frames, 1)) + \
              np.tile(np.arange(0, num_frames * frame_step, frame_step), (frame_length, 1)).T
    ''' 
        Przypisanie tych wskaźników (mapy), do faktycznego sygnału - zamiana 1D na 2D
    '''
    frames = pad_signal[indices.astype(np.int32, copy=False)]
    ''' 
        Na koniec stosujemy funkcję hamminga w(n) = 0.54 - 0.46 * cos(2*pi*n / (N-1)),
        która wygładza (wycisza) brzegi aby było to bardziej przystępne dla FFT
    '''
    frames *= np.hamming(frame_length)

    return frames

def power_spectrum(frames, NFFT):
    ''' 
        Ta funkcja liczy fft ramek a potem ich widma mocy. Audio jest rzeczywiste więc używamy RFFT
        NFFT to liczba punktów FFT (zazwyczaj 256 lub 512).
    '''
    fft_frames = np.absolute(np.fft.rfft(frames, NFFT))
    pow_spec_frames = ((1.0 / NFFT) * (fft_frames ** 2))
    
    return pow_spec_frames

def get_mel_filter_banks(num_filters, NFFT, sr):
    ''' 
        Chcemy stworzyć te trójkątne filtry i je tutaj zastosować
    '''
    low_freq_mel = 0
    high_freq_mel = (2595 * np.log10(1 + (sr / 2) / 700))
    mel_points = np.linspace(low_freq_mel, high_freq_mel, num_filters + 2)
    '''
        Wykorzystaliśmy mel aby wybrać punkty dzielące na trójkąty, a teraz wracamy już do Hz
    '''
    hz_points = (700 * (10**(mel_points / 2595) - 1))
    ''' 
        Teraz potrzebujemy indeksy tablicy w których zaczynają się trójkąty, mają swój szczyt
        i się kończą. Te indeksy będą dotyczyć tablicy z punktami po zrobieniu FFT na danej ramce.
        
        Cała skala ma sr Hz (dla nas 22050Hz), a więc ułamek hz_points / sr oznacza jaką częścią
        całej skali jest dany dźwięk. 

        Następnie te ułamki mnożymy przez (NFFT + 1), czyli ilość dostępnych 'miejsc' w FFT oraz 
        zaokrąglamy w dół bo chcemy naturalne indeksy
    '''
    indices = np.floor((NFFT + 1) * hz_points / sr)
    ''' 
        Teraz będziemy wypełniać macierz. Zaczynamy od samych zer w docelowym kształcie:
        num_filters x NFFT/2 + 1, gdzie to drugie bierze się z specyfikacji RFFT, która odrzuca
        drugą połowę danych, które dla nas nie są potrzebne
    '''
    fbank = np.zeros((num_filters, int(np.floor(NFFT / 2 + 1))))
    for m in range(1, num_filters + 1):
        f_m_start = int(indices[m - 1])   # trójkąt się zaczyna 
        f_m_center = int(indices[m])      # szczyt (czubek)
        f_m_end = int(indices[m + 1])     # trójkąt się kończy 
        for k in range(f_m_start, f_m_center):
            # Prosta linia rosnąca między start a center - interpolacja tego od 0 do 1
            fbank[m - 1, k] = (k - indices[m - 1]) / (indices[m] - indices[m - 1])
        for k in range(f_m_center, f_m_end):
            # to samo tylko teraz między start a end - od 1 do 0
            fbank[m - 1, k] = (indices[m + 1] - k) / (indices[m + 1] - indices[m])
    
    return fbank

def compute_mfcc(audio, sr, num_filters=40, num_ceps=12, NFFT=512):
    signal = pre_emphasis(audio)
    frames = frame_and_window_signal(signal, sr)
    pow_spec = power_spectrum(frames, NFFT)
    fbank = get_mel_filter_banks(num_filters, NFFT, sr)
    ''' 
        Przemnażamy macierz z naszymi spektrami mocy (ramki x NFFT/2) z macierzą z filtrami (NFFT/2 x n_f)
        Dostajemy (ramki x num_filters)
        Na koniec zastępujemy 0 przez epsilon aby uniknąć liczenia log(0)
        Potem logarytmujemy i dostajemy wartości w dB
    '''
    filter_banks = np.dot(pow_spec, fbank.T)
    filter_banks = np.where(filter_banks == 0, np.finfo(float).eps, filter_banks)
    filter_banks = 20 * np.log10(filter_banks)
    ''' 
        Na koniec robimy na tej macierzy DCT 
        norm='ortho' sprawia że macierz jest ortogonalna

        Zwracamy num_ceps parametrów, zaczynając od tego drugiego, bo pierwszy jest tylko wskaźnikiem
        głośności. DCT układa nam 40 filtrów na czynniki pierwsze od najważniejszego, 
        do najmniej ważnego pod względem ich wag. Dlatego możemy pominąć potem pozostałe np. 27

        Zwracane mfcc to macierz (ramki x num_ceps)
    '''
    mfcc = scipy.fftpack.dct(filter_banks, type=2, axis=1, norm='ortho')
    return mfcc[:, 1 : (num_ceps + 1)]

# Cechy spektralne
## Słowem wstępu
Aby wyliczyć spectral centroid, contrast i rollof potrzebuje wartości magnitud w czasie - wylicza się to wykorzystując algorytm FFT

### Standardowy FFT
FFT to algorytm który mając surowy sygnał wyliczy nam z jakich częstotliwości składa się sygnał

![fft](images/fft.png)

Problem - jeśli wrzucimy do tego algorytmu calą piosenke(sygnał) FFT wypisze nam częstotliwości wystepujące w całym utworze - bez podziału na kolejność

### Rozwiązanie STFT
Intuicja - dzielimy sygnał na okna czasowe i dla kazdego takiego przedziału wyliczamy FFT otrzymując macierz częstotliwosc x czas

In [23]:
def stft(audio_data,sr=22050, n_fft=2048, hop_length=512):
    """
    Ręczna implementacja STFT.
    input:
    - nfft: liczba próbek w kadej ramce, wieksze nfft - lepsza rozdzielczosc czestotliwosci ale gorsza czasowa, mniejsze nfft - na odwrót
    -hop_length: liczba próbek o które przesuwamy okno między kolejnymi ramkami
    output:
    - magnitudes: Macierz amplitud (częstotliwość x czas)
    - freq: tablica zakresu czestotliwosci
    """
    #1. Przygotowanie okna (Hanning Window) - statystyka: redukuje listki boczne (wyciek widma)
    window = 0.5 - 0.5 * np.cos(2 * np.pi * np.arange(n_fft) / n_fft)
    
    #2. Podział na ramki
    n_samples = len(audio_data)
    n_frames = 1 + (n_samples - n_fft) // hop_length
    
    # rfft zwraca n_fft/2 + 1 prążków częstotliwości (bo sygnał jest rzeczywisty, druga połowa to lustro)
    n_bins = n_fft // 2 + 1
    magnitudes = np.zeros((n_bins, n_frames))
    
    for i in range(n_frames):
        start = i * hop_length
        end = start + n_fft
        segment = audio_data[start:end]
        spectrum = np.fft.rfft(segment * window)
        
        #interesuje nas amplituda- wartość bezwzględna liczby zespolonej
        magnitudes[:, i] = np.abs(spectrum)

    freq = np.fft.rfftfreq(n_fft, d=1/sr)
        
    return magnitudes, freq

# Cechy barwowe:

### Spectral Centroid

To jest miara jasności dźwięku. Matematycznie jest to średnia ważona częstotliwości, gdzie wagami są amplitudy (energie) tych częstotliwości w danej chwili.

![eq](<images/spectral_centroid_eq.png>)
![desc](<images/spectral_centroid_desc.png>)


In [24]:
def compute_spectral_centroid(magnitudes,freq_bins):

    numerator = np.sum(magnitudes * freq_bins.reshape(-1, 1), axis=0)
    denominator = np.sum(magnitudes, axis=0)
    
    #dodajemy eps zeby na pewno nie podzielic przez zero
    eps = np.finfo(float).eps
    spectral_centroid = numerator / (denominator + eps)

    return np.array(spectral_centroid)

### Spectral Rolloff

Jest to częstotliwość Rt, poniżej której znajduje się określony procent (zazwyczaj 85%) całkowitej energii widma. Można to traktować jako kwantyl rzędu 0.85 rozkładu energii.

![eq](images/spectral_rolloff_eq.png)


In [25]:
def copmpute_spectral_rollof(magnitudes,freq_bins):
    threshold_percent = 0.85
    total_energy = np.sum(magnitudes, axis=0)
    threshold_energy = total_energy * threshold_percent
    
    #kumulujemy energię wzdłuż częstotliwości
    cumulative_energy = np.cumsum(magnitudes, axis=0)
    
    #szukamy indeksu, gdzie suma przekracza próg
    rolloff_indices = np.argmax(cumulative_energy >= threshold_energy, axis=0)
    spectral_rolloff = freq_bins[rolloff_indices]
    return np.array(spectral_rolloff)

### Spectral Contrast
    
Ta cecha dzieli widmo na pod-pasma (oktawy) i dla każdego pasma liczy różnicę między szczytami (peaks) a dolinami (valleys) energii.

In [26]:
def compute_spectral_contrast(magnitudes):
    #dziele widmo na 6 pasm i wyliczam dla kazdego pasma spectral contrast

    n_bands = 6
    n_bins = magnitudes.shape[0]
    band_size = n_bins // n_bands
    
    contrasts = []
    
    for i in range(n_bands):
        start = i * band_size
        end = (i + 1) * band_size
        band_magnitude = magnitudes[start:end, :]
        
        # Sortujemy amplitudy w paśmie, żeby znaleźć piki i doliny
        # Quantile method: alpha pika i alpha doliny
        peak = np.percentile(band_magnitude, 98, axis=0) # Górne 2%
        valley = np.percentile(band_magnitude, 2, axis=0) # Dolne 2%
        
        # Kontrast to różnica w skali logarytmicznej (dB)
        # Logarytmujemy, bo ludzkie ucho słyszy głośność logarytmicznie
        contrast = np.log1p(peak) - np.log1p(valley)
        contrasts.append(np.mean(contrast)) #średnia kontrastu w tym paśmie dla całego utworu

    return np.array(contrasts)



# Test

In [27]:
with open('dataset.pkl','rb') as f:
    dataset = pickle.load(f)

first_file_key = list(dataset.keys())[0]

magnitudes, freq = stft(dataset[first_file_key])

spectral_centroid = compute_spectral_centroid(magnitudes,freq)
spectral_rolloff = copmpute_spectral_rollof(magnitudes,freq)
spectral_contrast = compute_spectral_contrast(magnitudes)


print(spectral_centroid,spectral_centroid.shape)
print(spectral_rolloff,spectral_rolloff.shape)
print(spectral_contrast,spectral_contrast.shape)

[947.09066764 915.71470635 964.50292642 ...   0.           0.
   0.        ] (10358,)
[1636.5234375  1571.92382812 1604.22363281 ...    0.            0.
    0.        ] (10358,)
[3.45988111 1.76426232 1.11004561 0.951908   0.9161976  0.74743681] (6,)


# Problem do rozwiazania!

Kazda z tych cech jest wektorem liczb zaleznym od dlugosci wektora wejściowego (surowego sygnalu mp3) - z tego powodu w sumie to sa to trajektorie bardziej niz wektory ale whatever

Pierwsza propozycja naprawcza (dosyć prymitywna ale moze nie glupia) - dla kazdej takiej trajektorii liczymy srednia i wariancje i thats it

# Zero Crossing Rate
Zero Crossing Rate to po prostu liczba zmian znaku sygnału w określonym czasie. Mówiąc inaczej: liczymy, ile razy wykres fali dźwiękowej przecina oś X

## Wzór
$$ZCR = \frac{1}{T-1} \sum_{t=1}^{T-1} \mathbb{I}(s_t s_{t-1} < 0)$$

- $s_t$ to wartość próbki w czasie $t$
- $\mathbb{I}$ to funkcja indykator, która zwraca 1, jeśli warunek jest prawdziwy (zmiana znaku), i 0 w przeciwnym wypadku.

## Użyteczność
ZCR jest bardzo prostym, ale potężnym wskaźnikiem "zaszumienia" lub "perkusyjności" dźwięku.

1. Rozróżnianie dźwięków tonalnych od szumu:
    - Niskie ZCR: Sygnały, które zmieniają znak rzadko, są zazwyczaj "gładkie" i okresowe. Są to dźwięki tonalne (np. bas, wiolonczela, wokal w niskim rejestrze).

    - Wysokie ZCR: Sygnały, które szaleją i ciągle przecinają zero, są zazwyczaj szumem lub dźwiękami wysokoczęstotliwościowymi (np. talerze perkusyjne, hi-haty, spółgłoski szumiące jak "sz", "cz", "s").

2. Klasyfikacja gatunków: Muzyka rockowa/metalowa (dużo przesterowanych gitar i talerzy) będzie miała średnio wyższe ZCR niż muzyka klasyczna (skrzypce, fortepian).

In [28]:
def calculate_zcr(audio, size=2048, step=512):
    # Ile ramek zmieści się w audio?
    num_frames = 1 + (len(audio) - size) // step
    frames = np.array([audio[i * step : i * step + size] for i in range(num_frames)])

    # Wyciągamy znaki wszystkich próbek (-1, 0, lub 1)
    signs = np.sign(frames)
    # Mnożymy je. Jeśli wynik jest ujemny, znaczy że znaki były różne (+ * - = -)
    differences = signs[:, :-1] * signs[:, 1:] < 0
    zcr_counts = np.sum(differences, axis=1)
    # Normalizujemy dzieląc przez długość ramki 
    zcr_normalized = zcr_counts / size
    
    return zcr_normalized

# Cechy Harmoniczne

### Chroma features
mówi nam jak bardzo któryś z półtonów (C, C#, D, D#, E, F, F#, G, G#, A, A#, B) jest obecny w danym momencie utworu. Ignorujemy oktawy

Zwracamy macierz 12xT -> dla każdego okienka czasowego mamy wektor, mówiący jak bardzo któryś z półtonów jest obecny

Aby zamienić częstotliwość (Hz) na numer MIDI, używamy wzoru:

$$
m = 69 + 12 \cdot \log_2\left(\frac{f}{440}\right)
$$

gdzie:
- $f$ — częstotliwość w Hz,
- 69 — numer MIDI dla A4 (440 Hz),
- $\log_2$ — logarytm o podstawie 2 (liczba oktaw względem A4),
- mnożenie przez 12 zamienia oktawy na półtony.

In [29]:
def compute_chroma(magnitudes, freqs):
    n_bins, n_frames = magnitudes.shape

    # Liczymy czestotliwosci MIDI dla wszystkich czestotliwosci na raz
    m = 69 + 12 * np.log2(freqs[1:] / 440.0)
    chroma_bins = np.round(m).astype(int) % 12  # 12 poltonow
    chroma_matrix = np.zeros((12, n_frames))

    for t in range(n_frames):
        np.add.at(chroma_matrix[:, t], chroma_bins, magnitudes[1:, t])

    chroma_matrix /= np.sum(chroma_matrix, axis=0, keepdims=True) + 1e-9

    return chroma_matrix

# --------------------------------------

# Cechy Rytmiczne

### Tempo - BPM
Liczba uderzeń na minutę, obliczamy tempo na podstawie STFT

$
\text{BPM} = \frac{60}{\text{czas między uderzeniami [s]}}
$

In [30]:
def estimate_bpm(magnitudes, sr=22050, hop_length=512, min_bpm=60, max_bpm=200):
    # ile energii mamy w danym okresie czasowym?
    energy = np.sum(magnitudes, axis=0)
    energy = (energy - np.mean(energy)) / (np.std(energy) + 1e-9) # normalizacja

    # porównujemy sygnał ze sobą po jakimś przesunięciu, sprawdzamy po jakim przeusnięciu
    # korelacja jest największa
    autocorr = np.correlate(energy, energy, mode='full')
    autocorr = autocorr[autocorr.size//2:]  # bierzemy tylko dodatnie lag

    # ograniczamy bpm do sensownego przedziału
    min_lag = int(sr * 60 / max_bpm / hop_length)
    max_lag = int(sr * 60 / min_bpm / hop_length)
    peak_index = np.argmax(autocorr[min_lag:max_lag]) + min_lag

    # zamieniamy na bpm
    period_seconds = peak_index * hop_length / sr
    bpm = 60 / period_seconds
    return bpm

### Tempogram
Zamiast jednej liczby tempa jak w bpm na cały utwór, tutaj mamy informacje jak tempo zmienia się w czasie trwania utworu. 

Zwracamy rozkład siły różnych temp w różnych odcinkach czasowych dzięki czemu możemy uchwycić współistnienie kilku temp i ich zmiany w czasie

In [31]:
def compute_tempogram(magnitudes, sr=22050, hop_length=512, window_size=128, hop=32, min_bpm=60, max_bpm=200):
    
    energy = np.sum(magnitudes, axis=0)
    energy = (energy - np.mean(energy)) / (np.std(energy) + 1e-9) # normalizacja

    min_lag = int(sr * 60 / max_bpm / hop_length)
    max_lag = int(sr * 60 / min_bpm / hop_length)

    tempos = []
    
    for start in range(0, len(energy) - window_size, hop):
        window = energy[start:start + window_size]

        # porównujemy sygnał ze sobą po jakimś przesunięciu, jeśli wartość jest wysoka, to 
        # najprawdopodobniej mamy jakiś dzwięk z takim bpm
        autocorr = np.correlate(window, window, mode='full')
        autocorr = autocorr[autocorr.size // 2:]

        local_autocorr = autocorr[min_lag:max_lag]
        # local_autocorr[k] - jak bardzo rytm w tym fragmencie pasuje do tempa odpowiadajacego indeksowi
        tempos.append(local_autocorr)

    return np.array(tempos).T

In [32]:
chroma = compute_chroma(magnitudes, freq)
bpm = estimate_bpm(magnitudes)
tempogram = compute_tempogram(magnitudes)
print("CHROMA:")
print(chroma, chroma.shape)
print(f"BMP Utworu: {bpm}\nTEMPOGRAM")
print(tempogram, tempogram.shape)

CHROMA:
[[0.08811088 0.11311137 0.16573456 ... 0.         0.         0.        ]
 [0.05842164 0.04297366 0.04519653 ... 0.         0.         0.        ]
 [0.09457696 0.05834978 0.06049996 ... 0.         0.         0.        ]
 ...
 [0.1074309  0.06592695 0.07397389 ... 0.         0.         0.        ]
 [0.07410802 0.08207489 0.06574852 ... 0.         0.         0.        ]
 [0.10216408 0.1839258  0.16202149 ... 0.         0.         0.        ]] (12, 10358)
BMP Utworu: 86.1328125
TEMPOGRAM
[[ 61.4338152   66.63043053  48.2993056  ... 208.6663486  221.63066151
  260.50993399]
 [ 61.84184965  66.31260161  47.92627021 ... 205.93695932 219.60591886
  257.91206513]
 [ 62.18693049  65.90423544  47.74200708 ... 203.2417427  217.47598928
  255.33490267]
 ...
 [ 45.5885449   45.6510345   32.76493988 ... 154.29975233 170.58189416
  190.69907627]
 [ 45.30770812  45.46461939  32.15324223 ... 152.60465346 168.4667972
  188.24653596]
 [ 45.54852315  45.19399303  31.69036171 ... 150.8409463  166.21

# Podsumowanie - pipeline

In [ ]:
from tqdm import tqdm

def song_to_representation(audio_data, sr=22050):
    feature_vector = []

    mfccs = compute_mfcc(audio_data, sr)
    feature_vector.extend(np.mean(mfccs, axis=0)) 
    feature_vector.extend(np.std(mfccs, axis=0))  

    zcr = calculate_zcr(audio_data)
    feature_vector.append(np.mean(zcr)) 
    feature_vector.append(np.std(zcr))  

    magnitudes, freqs = stft(audio_data, sr=sr)

    centroid = compute_spectral_centroid(magnitudes, freqs)
    feature_vector.append(np.mean(centroid))
    feature_vector.append(np.std(centroid))

    rolloff = copmpute_spectral_rollof(magnitudes, freqs) 
    feature_vector.append(np.mean(rolloff))
    feature_vector.append(np.std(rolloff))

    contrast = compute_spectral_contrast(magnitudes)
    feature_vector.extend(contrast) 

    chroma = compute_chroma(magnitudes, freqs)
    feature_vector.extend(np.mean(chroma, axis=1)) 
    feature_vector.extend(np.std(chroma, axis=1))  

    bpm = estimate_bpm(magnitudes, sr=sr)
    feature_vector.append(bpm) 

    tempogram = compute_tempogram(magnitudes, sr=sr)
    feature_vector.extend(np.mean(tempogram, axis=1)) 
    feature_vector.extend(np.std(tempogram, axis=1))

    # Konwersja na tablicę numpy float32 (lepsza dla ML)
    return np.array(feature_vector, dtype=np.float32)


def create_feature_dataset(input_pickle_path):
    """
    Wczytuje dataset audio (słownik), mieli go przez pipeline 
    i zwraca macierz X (cechy) oraz listę nazw plików (identyfikatory).
    """
    with open(input_pickle_path, 'rb') as f:
        raw_dataset = pickle.load(f)
    
    X = []
    filenames = []
    albums = []
        
    for filename, item in tqdm(raw_dataset.items()):
        try:
            if isinstance(item, dict):
                audio_data = item['audio']
                album_name = item.get('album', 'Unknown')
                if album_name is None:
                    album_name = 'Unknown'
            else:
                audio_data = item
                album_name = 'Unknown'

            features = song_to_representation(audio_data)
            # Sprawdzenie czy nie ma NaN lub Inf (częste przy logarytmach/dzieleniu przez 0)
            if np.isnan(features).any() or np.isinf(features).any():
                features = np.nan_to_num(features)
                
            X.append(features)
            filenames.append(filename)
            albums.append(album_name) 
        except Exception as e:
            print(f"Błąd przetwarzania {filename}: {e}")
            
    return np.array(X), filenames, albums

# TEST
df, fnames = create_feature_dataset("dataset.pkl")

  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:02<00:00,  1.06s/it]


In [34]:
print(df)

[[-2.85990906e+01 -1.19941387e+01 -7.23394823e+00  1.51187134e+00
  -1.57213306e+01 -1.16819878e+01 -1.90217927e-01 -1.17727222e+01
  -4.73757446e-01 -6.07175589e+00 -1.50662267e+00 -6.49953544e-01
   5.92280045e+01  3.47430382e+01  2.47584000e+01  2.13560524e+01
   2.18308773e+01  1.78047829e+01  1.61443939e+01  1.68324928e+01
   1.42180729e+01  1.43146849e+01  1.14505186e+01  1.13789034e+01
   9.26496908e-02  8.42311531e-02  2.27497314e+03  1.21662231e+03
   4.74717432e+03  2.39454834e+03  3.45988107e+00  1.76426232e+00
   1.11004567e+00  9.51907992e-01  9.16197598e-01  7.47436821e-01
   1.09893560e-01  7.49650225e-02  9.55624580e-02  6.65764064e-02
   8.15463960e-02  1.04466677e-01  7.24129528e-02  6.80657029e-02
   7.07010329e-02  9.48353857e-02  8.45761597e-02  7.42567033e-02
   4.14403267e-02  3.15309502e-02  3.69063690e-02  2.93485690e-02
   3.84829864e-02  5.03166392e-02  3.56456302e-02  2.93266829e-02
   3.22629735e-02  3.70957851e-02  3.71700339e-02  3.11544277e-02
   8.61328